# Step 31 Export BOLD Parcel PC1 With Nuisance Regression

## What This Notebook Does

This is the authoritative public Stage-3 export notebook for BOLD parcel time series.

It preserves the original exporter-side workflow that:
- freezes the atlas used for export,
- resamples that atlas onto each run's BOLD grid,
- performs voxelwise nuisance regression,
- extracts one parcel PC1 time series per Schaefer parcel,
- writes parcel outputs and run-level metadata,
- and writes the exporter QC sidecars used later for Table S5 and Figure S5.

Run this notebook before `step32_build_table_s4_bold_parcel_atlas_summary.ipynb` and `step33_build_table_s5_and_figure_s5_bold_qc.ipynb`.

## Manuscript Linkage

- Main Methods 2.3
- Supplementary Methods 1.4
- Supplementary Results 2.4
- Supplementary Tables S4-S5
- Supplementary Figure S5 support

## Important Provenance Note

This notebook keeps the exporter-side `res-02` Schaefer atlas path authoritative for parcel export.

The separate atlas-on-BOLD-grid notebook remains useful for atlas-preservation QC and overlay provenance, but it is **not** used here as the export-path atlas source.

## Inputs Expected

- fMRIPrep BOLD NIfTI files in template space
- matching brain masks
- fMRIPrep confounds TSV and JSON sidecars
- TemplateFlow Schaefer atlas files

## Outputs Written

- `dataset_index.csv`
- parcel PC1 NPY and MAT outputs
- exporter-side `atlas_on_grid` NIfTI files
- `motion_qc_flags.csv`
- `qc_motion_to_pc.csv`
- `qc/qc_parcel_blowups.csv`
- per-run QC PNGs


In [ ]:
# Step 0. Import Python modules and locate the Stage-3 helper file

import sys
from pathlib import Path

STAGE3_DIR = Path.cwd()
if not (STAGE3_DIR / "stage3_bold_export_helpers.py").exists():
    candidate = Path.cwd() / "notebooks" / "3_bold"
    if candidate.exists():
        STAGE3_DIR = candidate

if str(STAGE3_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE3_DIR.resolve()))

from stage3_bold_export_helpers import run_bold_parcel_export


In [ ]:
# Step 1. User-editable root folders and run options
#
# Replace the <SET_...> placeholders with paths on your machine.
# The atlas settings below preserve the recovered final export branch.

BIDS_ROOT = Path("<SET_BIDS_ROOT>")
DERIVATIVES_ROOT = Path("<SET_DERIVATIVES_ROOT>")

PARCEL_OUTPUT_FOLDER = "parcel_pc1_v6"
ATLAS_SPACE = "MNI152NLin2009cAsym"
ATLAS_NAME = "Schaefer2018"
ATLAS_DESC = "200Parcels7Networks"
ATLAS_RESOLUTION = 2
N_PARCELS = 200

BOLD_PREFERENCE = [
    f"*_task-rest_*space-{ATLAS_SPACE}_desc-preproc_bold.nii.gz",
]

USE_MOTION24 = True
USE_WM_CSF = True
USE_COSINES = True
N_ACOMPCOR = 10
ADD_NONSTEADY = True
ADD_FD_SPIKES = True
ADD_DVARS_SPIKES = False
FD_THRESHOLD = 0.5
DVARS_ZTHRESH = 2.5
BLOCK_MIN_LEN = 3
MIN_VOXELS_FOR_PCA = 10
MIN_VOXELS_FOR_ANY = 1
ZSCOR_BEFORE_PCA = True
SAVE_NPY = True
SAVE_MAT = True
SAVE_ATLAS_ON_GRID = True
SAVE_QC_FIGS = True
SAVE_QC_SIDECARS = True


In [ ]:
# Step 2. Build the derived paths and print a short run summary

FMRI_ROOT = BIDS_ROOT
OUT_ROOT = DERIVATIVES_ROOT / "bold_parcel" / PARCEL_OUTPUT_FOLDER

def assert_configured_path(path_value, label):
    path_str = str(path_value)
    if "<SET_" in path_str:
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")

assert_configured_path(BIDS_ROOT, "BIDS_ROOT")
assert_configured_path(DERIVATIVES_ROOT, "DERIVATIVES_ROOT")

print("Stage 3 / Step 31: export BOLD parcel PC1 with nuisance regression")
print(f"  fMRIPrep root:        {FMRI_ROOT}")
print(f"  Output root:          {OUT_ROOT}")
print(f"  Atlas space:          {ATLAS_SPACE}")
print(f"  Atlas desc:           {ATLAS_DESC}")
print(f"  Atlas resolution:     res-{ATLAS_RESOLUTION:02d}")
print(f"  N parcels:            {N_PARCELS}")
print(f"  FD threshold:         {FD_THRESHOLD}")
print(f"  Write QC sidecars:    {SAVE_QC_SIDECARS}")


In [ ]:
# Step 3. Run the preserved exporter helper and review the run summary table

qc_df = run_bold_parcel_export(
    fmri_root=FMRI_ROOT,
    out_root=OUT_ROOT,
    atlas_space=ATLAS_SPACE,
    atlas_name=ATLAS_NAME,
    atlas_desc=ATLAS_DESC,
    atlas_resolution=ATLAS_RESOLUTION,
    n_parcels=N_PARCELS,
    bold_preference=BOLD_PREFERENCE,
    use_motion24=USE_MOTION24,
    use_wm_csf=USE_WM_CSF,
    use_cosines=USE_COSINES,
    n_acompcor=N_ACOMPCOR,
    add_nonsteady=ADD_NONSTEADY,
    add_fd_spikes=ADD_FD_SPIKES,
    add_dvars_spikes=ADD_DVARS_SPIKES,
    fd_threshold=FD_THRESHOLD,
    dvars_z_thresh=DVARS_ZTHRESH,
    block_min_len=BLOCK_MIN_LEN,
    min_voxels_for_pca=MIN_VOXELS_FOR_PCA,
    min_voxels_for_any=MIN_VOXELS_FOR_ANY,
    zscore_before_pca=ZSCOR_BEFORE_PCA,
    save_npy=SAVE_NPY,
    save_mat=SAVE_MAT,
    save_atlas_on_grid=SAVE_ATLAS_ON_GRID,
    save_qc_figs=SAVE_QC_FIGS,
    save_qc_sidecars=SAVE_QC_SIDECARS,
)

qc_df.head()
